# **Visualizing Model Predictions & Confusion Matrices**

**Purpose:**  
This notebook allows you to:

- Save predicted segmentation masks generated by a trained model.
- Create and display a mosaic showing two RGB images alongside their corresponding predicted segmentations.
- Plot confusion matrices for the test set using the `history.json` file from a previous test run located in the log folder.

### **Notebook Structure:**

**A. Setup and Parameters:**

* Import the required libraries.

**B. Plot and Save Test RGB Images and Mosaic:**

* Display side-by-side pairs of RGB images and their corresponding segmentation results from the test set.
* Generate a 2×2 mosaic containing two RGB images and their predicted segmentations.

**C. Plot Confusion Matrices:**

* Load the `history.json` file from the most recent test run.
* If the test set does not contain a confusion matrix, display a recommendation in English explaining how to run tests to produce it.


### **Prerequisites:**

##### **Model Predictions:**

* You must have a well-defined `.ini` file inside the `/cfg` folder to provide the necessary information for model loading.
* You must have already trained the chosen network using our framework and know its log folder, since the trained weights produced during training will be used for model inference.
* You must set this folder's path in the `log` parameter of your `.ini` file, as shown in the example:
  `"logs= /logs/rellis3d_mobilevit_query_transformer_20250726-235549"`

##### **Confusion Matrices:**

* You must have already run the **test** phase of your pipeline to produce predicted masks.
* Inside the log folder of your trained network, there must be a `history.json` file containing your train and test results after running them.
* For confusion matrices: keep the resulting `history.json` path at hand and set it in the code.

For testing:

* Typical command to run the test (from your framework):

```bash
python run.py --cfg cfg/rellis3d_dev.ini
```

**Make sure inside the .ini you set:**
`mode = test`

---

## **Visualizing Model Predictions**

### **1) Imports and basic setup**

In [ ]:
import os
import shutil
import configparser
from typing import Dict, Tuple
import warnings
import torch
from torchinfo import summary
from tqdm import tqdm
import numpy as np 
import json
import cv2
import matplotlib.pyplot as plt
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=UserWarning)
    
    import albumentations as A
    from albumentations.pytorch import ToTensorV2

from utils.utils import split_config
from utils.color import printH
from utils.segmentnet import SegmentNet
from utils.dataloaders import get_dataset
from utils.utils import color_image
from utils.utils import hex_to_rgb


import seaborn as sns

### **2) Load configuration (.ini)**

In [ ]:
config_path = "/home/lrm/workspace/segment_net/cfg/rellis3d_5090_segformerb0.ini"
if not os.path.exists(config_path):
        raise FileNotFoundError(("config file not found!"
                                    f"(cfg:{config_path})"))
        
config = configparser.ConfigParser()
config.read(config_path)    
config = config._sections   

for key, param in config.get("DIRS").items():
    if not os.path.exists(param):
        raise FileNotFoundError((f"{key} not found!"
                                    f"({key}:{param})"))

config = split_config(config)

with open (config.get("dirs").get("mapping"), 'r') as f:
        mapping = json.load(f)

        #{name, num} -> {num, name}
        class_id_mapping_image      = {v:k for k, v in mapping.get("target_classes").items()}

        color_mapping_image = {}
        for k, v in mapping.get("target_color").items():
            if mapping.get("target_classes").get(k) not in color_mapping_image:
                color_mapping_image[mapping.get("target_classes").get(k)] = hex_to_rgb(v.replace("#",""))

### **3) Get log informations and load model**

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = SegmentNet(config=config)
model.to(device)

log_dir = config.get("dirs").get("logs")

if not os.path.exists(log_dir):
    raise FileNotFoundError(f"Checkpoint folder not found! (log_dir:{log_dir})")
        
printH("[Image Segmentation][main]", f"log_dir: {log_dir}", "i")

checkpoint_path = os.path.join(log_dir, "best_model.pth")
    
if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(f"Checkpoint file not found! (checkpoint:{checkpoint_path})")
        
printH("[Image Segmentation][main]", f"checkpoint: {checkpoint_path}", "i")
                
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.to(device)


printH("[Image Segmentation][main]", f"model summary:", "i")
w, h = config.get("image").get("image_size")  
summary(model, input_size=(1, 3, h, w))

[SegmentNet] init...
[Image Segmentation][main] log_dir: /home/lrm/workspace/segment_net/logs/rellis3d_segformerb0_query_transformer_20250727-203945
[Image Segmentation][main] checkpoint: /home/lrm/workspace/segment_net/logs/rellis3d_segformerb0_query_transformer_20250727-203945/best_model.pth
[Image Segmentation][main] model summary:


Layer (type:depth-idx)                                                           Output Shape              Param #
SegmentNet                                                                       [1, 20, 400, 640]         --
├─SegFormerB0_FPN: 1-1                                                           [1, 128, 128, 128]        --
│    └─SegformerModel: 2-1                                                       [1, 32, 128, 128]         --
│    │    └─SegformerEncoder: 3-1                                                [1, 32, 128, 128]         3,319,392
│    └─FeaturePyramidNetwork: 2-2                                                [1, 128, 16, 16]          --
│    │    └─ModuleList: 3-8                                                      --                        (recursive)
│    │    └─ModuleList: 3-9                                                      --                        (recursive)
│    │    └─ModuleList: 3-8                                                      --       

### **4) Create folder to save images**

In [ ]:
result_dir = config.get("dirs").get("logs")
result_dir = os.path.join(result_dir, "inference_images")

if os.path.exists(result_dir):
    shutil.rmtree(result_dir)

os.makedirs(result_dir)

### **5) Load Data and define saving path**

In [ ]:
#[train, val, test]
subset_name = "test"

w, h = config.get("image").get("image_size")  
    
transform = A.Compose([
            A.Resize(height=h, width=w),
            ToTensorV2()
        ])

data = get_dataset(
        config=config,
        set_name=subset_name,
        num_samples=-1,
        transform=transform,
        return_data_path=True
)

root_path_images = os.path.join(result_dir, subset_name)
if os.path.exists(root_path_images):
    shutil.rmtree(root_path_images)

os.makedirs(root_path_images)

printH(f"[Image Segmentation][visualization][{subset_name}]", f"images will be saved at: {root_path_images}", "i")

[Rellis-3D Dataset][test] creating dataloader...
[Rellis-3D Dataset][test] loaded the metadata!


100%|██████████| 1672/1672 [00:00<00:00, 270788.33it/s]

[Rellis-3D Dataset][test] found 1672 samples!
[Image Segmentation][visualization][test] images will be saved at: /home/lrm/workspace/segment_net/logs/rellis3d_segformerb0_query_transformer_20250727-203945/inference_images/test


### **6) Getting the RGB image from a tensor format**

In [ ]:
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

def tensor_to_rgb_image(t: torch.Tensor, mean=MEAN, std=STD):
    t = t.detach().cpu().float()
    if t.ndim == 4:
        t = t[0]
    for c in range(3):
        t[c] = t[c] * std[c] + mean[c]
    img = t.clamp(0, 1).numpy().transpose(1, 2, 0)
    return (img * 255.0).round().astype(np.uint8) 


### **7) Inference and Visualization**

In [ ]:

samples_for_mosaic = []

model.eval()
for (x_img, y_img, z_img) in tqdm(data):
    image_name = z_img.split("/")[-1]
    image_path = os.path.join(root_path_images, image_name)

    x_img, y_img = x_img.to(device), y_img.to(device)
    x_img = x_img.unsqueeze(0)

    y_pred_img = model.predict(x_img, normalize=True)
    y_pred_img = y_pred_img.cpu().numpy()

    y_pred_img = [color_image(y_pred_img[i], color_mapping_image) for i in range(y_pred_img.shape[0])]
    y_pred_img = np.asarray(y_pred_img)[0].astype(np.uint8) 

    y_pred_bgr = cv2.cvtColor(y_pred_img, cv2.COLOR_RGB2BGR)
    cv2.imwrite(image_path, y_pred_bgr)

    if len(samples_for_mosaic) < 2:
        rgb_input_u8 = tensor_to_rgb_image(x_img) 
        samples_for_mosaic.append((rgb_input_u8, y_pred_img))


if len(samples_for_mosaic) == 2:
    (rgb1, pred1), (rgb2, pred2) = samples_for_mosaic

    plt.figure(figsize=(12, 10))

    ax = plt.subplot(2, 2, 1)
    ax.imshow(rgb1); ax.set_title("RGB Image #1", fontweight="bold"); ax.axis("off")

    ax = plt.subplot(2, 2, 2)
    ax.imshow(pred1); ax.set_title("Predicted Segmentation #1", fontweight="bold"); ax.axis("off")

    ax = plt.subplot(2, 2, 3)
    ax.imshow(rgb2); ax.set_title("RGB Image #2", fontweight="bold"); ax.axis("off")

    ax = plt.subplot(2, 2, 4)
    ax.imshow(pred2); ax.set_title("Predicted Segmentation #2", fontweight="bold"); ax.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("Not enough samples to build the 2×2 mosaic (need 2).")

---

## **Plotting Confusion Matrix**

### **1) Define the path for history.json file**

In [ ]:
json_path = "/home/lrm/workspace/segment_net/logs/rellis3d_maxxvitv2_query_transformer_20250802-185128/history.json"

### **2) Load and prepare confusion matrix plotting utilities**

In [ ]:

with open(json_path, 'r') as f:
    data = json.load(f)

results_dir = "./results_cm"
os.makedirs(results_dir, exist_ok=True)

def process_confmat(confmat_data):
    shape = confmat_data['shape']
    flat_data = confmat_data['data']
    cm = np.array(flat_data).reshape(shape)

    row_sums = cm.sum(axis=1, keepdims=True)
    cm_normalized = np.divide(cm, row_sums, where=row_sums!=0)

    return cm_normalized

def plot_confusion_matrix(cm, labels, title, filename):
    fig_size = (10, 8)
    font_size = 14

    plt.figure(figsize=fig_size)
    
    annot = np.where(cm >= 0.04, np.char.mod('%.2f', cm), '')

    ax = sns.heatmap(cm, annot=annot,
                     cmap='YlOrRd',
                     xticklabels=labels,
                     yticklabels=labels,
                     fmt="",
                     cbar=True,
                     vmin=0, vmax=1,
                     annot_kws={"fontsize": font_size - 2})

    plt.xlabel('Predicted', fontsize=18, fontweight='bold')
    plt.ylabel('True', fontsize=18, fontweight='bold')
    plt.gca().xaxis.set_label_coords(0.5, 1.05)
    plt.gca().yaxis.set_label_coords(-0.07, 0.5)
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=font_size)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=font_size)

    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, filename))
    plt.show()
    plt.clf()


### **3) Process and plot confusion matrix for tests**

In [ ]:

sample_cm = data['test']['image']['confmat'][0]
num_classes = sample_cm['shape'][0]
lat_labels = list(range(num_classes))

# You can also adpt it to plot your matrices for train and validation 
modes = {
    'test': data.get('test', {}).get('image', {}).get('confmat', [])
}

for mode, confmat_list in modes.items():
    if confmat_list:
        cm_data = confmat_list[0]  
        cm_norm = process_confmat(cm_data)

        classes_to_keep = list(range(2, cm_norm.shape[0]))  
        cm_norm = cm_norm[np.ix_(classes_to_keep, classes_to_keep)]
        lat_labels_filtered = classes_to_keep

        plot_confusion_matrix(cm_norm, lat_labels_filtered, f"{mode.upper()} - Confusion Matrix", f"{mode}_cm.pdf")
        print(f"{mode.upper()} matrix saved at: {os.path.join(results_dir, f'{mode}_cm.pdf')}")
    else:
        if mode == "test":
            print("\n⚠ No confusion matrix found for TEST mode.")
            print("To generate it, please run the pipeline in TEST mode using a configuration file.")
            print("Example:")
            print("    python run.py --cfg cfg/rellis3d_dev.ini")
            print("Make sure that inside the .ini file you set:")
            print("    mode = test\n")
        else:
            print(f"No matrix found for '{mode}'.")
